# ColabDFT SPE — DFT の一点計算

入力した構造のまま、[GPU4PySCF](https://github.com/pyscf/gpu4pyscf) で **DFT の一点計算（SPE）** を行うノートブックです。構造の最適化はしません。コードを書く必要はありません。UMA は使わないので、Hugging Face のトークンはいりません。

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 下の **1. Input form** で出力名、電荷、スピン多重度、汎関数、基底関数を指定する
3. **ランタイム → すべて実行** を押す
4. **2. File selection and checks** にファイル選択ボタンが出るので、xyz または Gaussian の gjf / com を選ぶ。あとは放置できます
5. 下のセルで 3D 表示とエネルギーを確認し、最後のセルで結果のZIPをダウンロードする

gjfからは構造だけを読み込みます。計算方法はフォームの値を使い、電荷と多重度もフォームの値を使います。xyz に複数の構造が入っている場合は先頭だけを計算します。

2回目以降は手順2〜5を繰り返すだけです（インストールは自動でスキップされます）。ファイル選択で Cancel upload を押すと、前回のファイルをそのまま使います。

## 出力（ZIPの中身）

| ファイル | 内容 |
|---|---|
| `<出力名>_spe.xyz` | 計算した構造。コメント行にエネルギーと計算条件 |
| `<出力名>_dft.csv` | DFT 一点計算のエネルギー |
| `<出力名>_log.txt` | 入力条件、ソフトのバージョン、計算結果 |

## 注意事項

- DFT は GPU4PySCF（密度フィッティング）で計算します。T4 は倍精度の計算が遅いため、原子数が多いときや def2-TZVPD では時間がかかります。GPU が使えないときは CPU の PySCF で計算します。
- 閉殻（多重度1）は RKS、開殻は UKS で計算します。
- 扱える元素は H〜Rn（原子番号1〜86）です。周期系（結晶・表面）には対応していません。
- UMA で構造を最適化してから一点計算するときは [ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Opt.ipynb) を使ってください。
- 静電ポテンシャル（ESP）を見るときは [ColabDFT_ESP](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabDFT_ESP.ipynb) を使ってください。

## ライセンス

- このノートと共通コード（`colabuma`）：[MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- 実行時にインストールするライブラリは、それぞれのライセンスに従います。一覧は [README のライセンス](https://github.com/RChibaChiba/ColabChem/blob/main/README.ja.md#ライセンス) にあります

## 引用

論文で使う場合は、以下を引用してください。

- PySCF（Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020)）
- GPU4PySCF（X. Wu *et al.*, arXiv:2404.09452 (2024)）

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### DFT single point (gpu4pyscf)
#@markdown ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection happens here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, output, structure, ui
from colabuma.ui import ok

ui.install_error_handler()

structure.check_multiplicity(multiplicity)
functional, basis = dft.parse_method(dft_functional, dft_basis)
job = dft.Job(structure.out_prefix(output_name), charge, multiplicity, functional, basis)
env.check_gpu()
upload_name, upload_data = structure.upload()
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install) { display-mode: "form" }
DEVICE = env.setup_dft()

In [ ]:
#@title 4. Prepare the structure { display-mode: "form" }
atoms, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity, uma=False)
structure.show_numbered(atoms)
ok(f"Structure ready: {atoms.get_chemical_formula()} ({len(atoms)} atoms).")

In [ ]:
#@title 5. DFT single point { display-mode: "form" }
dft_res = dft.run([("Input structure", atoms)], charge, multiplicity, functional, basis)
files, dft_log = dft.report(dft_res, None, geometry="input structure")
_p = dft_res["points"][0]
files["spe.xyz"] = structure.xyz_block(atoms.get_chemical_symbols(), atoms.positions,
                                       f"energy_Ha={_p['energy']:.10f} converged={_p['converged']} "
                                       f"method={functional}/{basis} charge={charge} multiplicity={multiplicity}")

In [ ]:
#@title 6. Save { display-mode: "form" }
files["log.txt"] = output.log_text("ColabDFT SPE log", source_desc, job, [dft_log])
zip_path = output.save_zip(job.out_prefix, files)